# Lab 9: Working with Hierarchical Data: Tree Visualizations

## Objectives

By the end of this lab, you should be able to:

- Explain what makes data hierarchical, and how that differs from the flat, tabular shape used everywhere else this semester
- Reshape grouped, aggregated data into the structure tree charts expect
- Build and interpret a treemap and a sunburst chart, and choose between them for a given hierarchy
- Use tree charts' built-in click-to-drill-down interaction as a continuation of Lab 7's interactive EDA skills

## Setup

We'll use a retail sales dataset for this lab: individual orders from a superstore chain, with each order tagged by region, product category, and product sub-category.

In [ ]:
# !pip install pandas plotly

import pandas as pd
import plotly.express as px

sales = pd.read_csv("superstore.csv")
sales.head()

,region,category,sub_category,segment,sales,profit,quantity,order_date
0,Nunavut,Office Supplies,Storage & Organization,Small Business,261.5400,-213.25,6,10/13/2010
1,Nunavut,Office Supplies,Appliances,Consumer,10123.0200,457.81,49,10/1/2012
2,Nunavut,Office Supplies,Binders and Binder Accessories,Consumer,244.5700,46.71,27,10/1/2012
3,Nunavut,Technology,Telephones and Communication,Corporate,4965.7595,1198.97,30,7/10/2011
4,Nunavut,Office Supplies,Appliances,Consumer,394.2700,30.94,19,8/28/2010


## 1. What makes data hierarchical?

Every dataset we've used so far this semester has been flat: rows and columns, with no notion that one category contains another. Hierarchical data is different. It has **levels of nesting**, where each level belongs entirely inside the level above it.

Some familiar examples:

- A **file system**: a folder contains files and other folders, which contain more files and folders.
- An **org chart**: a company contains divisions, which contain teams, which contain individual employees.
- A **taxonomy**: a kingdom contains phyla, which contain classes, which contain species.

Our dataset today has a hierarchy too, even though it arrived as a flat table: every order belongs to a **region**, within that region to a **product category**, and within that category to a **sub-category**. `region -> category -> sub_category` is a three-level hierarchy hiding inside a flat CSV, and building a tree chart means making that structure explicit.

## 2. Preparing hierarchical data

The plotting functions in this lab expect a "path plus value" structure: one or more columns describing the nesting path, and one numeric column describing the size of each leaf. This is the same aggregation skill from Lab 2, just aimed at a new destination shape instead of a wide table.

Let's start with the simplest version: total sales by region.

In [ ]:
# TODO: group the sales data by region and calculate the total sales for each region
region_sales = sales.groupby("region", as_index=False).agg(total_sales=("sales", "sum"))
region_sales.sort_values("total_sales", ascending=False)

,region,total_sales
6,West,3.597549e+06
3,Ontario,3.063212e+06
4,Prarie,2.837305e+06
0,Atlantic,2.014248e+06
5,Quebec,1.510195e+06
7,Yukon,9.758674e+05
1,Northwest Territories,8.008473e+05
2,Nunavut,1.163765e+05


That's a familiar one-level summary. Now let's extend it to two levels, region and category, which is where the hierarchy starts to matter.

In [ ]:
# TODO: group the sales data by region and category, and calculate the total sales for each combination
region_category_sales = (
    sales
    .groupby(["region", "category"], as_index=False)
    .agg(total_sales=("sales", "sum"))
)
region_category_sales.head(10)

,region,category,total_sales
0,Atlantic,Furniture,7.087268e+05
1,Atlantic,Office Supplies,4.784644e+05
2,Atlantic,Technology,8.270570e+05
3,Northwest Territories,Furniture,2.828379e+05
4,Northwest Territories,Office Supplies,2.168489e+05
5,Northwest Territories,Technology,3.011605e+05
6,Nunavut,Furniture,4.354561e+04
7,Nunavut,Office Supplies,3.861547e+04
8,Nunavut,Technology,3.421540e+04
9,Ontario,Furniture,1.109618e+06


This table already has everything a tree chart needs: a path (`region`, then `category`) and a value (`total_sales`). The only thing left is to hand it to a chart type built to show that nesting visually instead of as rows.

## 3. Treemap: nested rectangles

A **treemap** represents each item in the hierarchy as a rectangle, with area encoding the value. Rectangles are nested inside their parent's rectangle, so the whole chart is one large rectangle subdivided down through every level of the hierarchy.

In [ ]:
# TODO: create a treemap of total sales by region and category, with the size of each rectangle representing the total sales
fig = px.treemap(
    region_category_sales,
    path=["region", "category"],
    width=800,
    height=600,
    values="total_sales",
    title="Total sales by region and category"
)
fig.show()

The `path` argument is doing the real work here: it tells Plotly which columns describe the nesting, from outermost to innermost. Larger rectangles mean more sales; you can compare rectangles both within a region (which category dominates there) and across regions (which region is largest overall) in a single glance, something that would take several separate bar charts to show as clearly.

Let's extend to the full three-level hierarchy by adding `sub_category`.

In [ ]:
#TODO: add a third level to the treemap by grouping the sales data by region, category, and sub-category
full_hierarchy = (
    sales
    .groupby(["region", "category", "sub_category"], as_index=False)
    .agg(total_sales=("sales", "sum"))
)

#TODO: Update treemap to include the sub-category level, with the size of each rectangle representing the total sales
fig = px.treemap(
    full_hierarchy,
    path=["region", "category", "sub_category"],
    values="total_sales",
    color="category",
    title="Total sales by region, category, and sub-category"
)
fig.show()

`color="category"` here encodes the same variable already shown by position in the hierarchy; that's a deliberate redundancy, called redundant encoding, and it makes it much easier to visually track which sub-categories belong to which category once there are many small rectangles on screen.

**Note:** If the data being plotted contains negative numbers, your chart will be unable to render because Plotly cannot biuld a mark with negative area for those rows. Always check here if your figure doesn't render.

## 4. Sunburst: the same hierarchy, radially

A **sunburst chart** shows the identical kind of data, path plus value, but arranges it as rings instead of nested rectangles: the center ring is the outermost level of the hierarchy, and each ring further out represents one level deeper.

In [ ]:
fig = px.sunburst(
    full_hierarchy,
    path=["region", "category", "sub_category"],
    values="total_sales",
    color="category",
    title="Total sales by region, category, and sub-category (sunburst)"
)
fig.show()

**Choosing between them:**

- **Treemap**: area is easier to compare accurately than the wedge shapes a sunburst uses, especially for leaves that are far apart in the hierarchy. Better when your main question is "which items are the biggest," across the whole dataset.
- **Sunburst**: the ring structure makes the *depth* of the hierarchy and the parent-child relationships more immediately visible, since every level gets its own visually distinct ring. Better when your main question is about structure itself, how many levels deep does this go, and how does one level relate to the level above it, rather than precise size comparison.

Look at both charts above. Which one makes it easier to answer "which single sub-category has the highest sales, out of everything"? Which makes it easier to answer "how many categories does the West region have, and how do they compare to each other specifically within that region"?

## 5. Built-in interactivity: click to drill down

Both chart types support the same interaction, continuing directly from Lab 7: **click any segment to zoom into that branch**, and click the chart's title area (or use the path breadcrumb Plotly adds automatically) to zoom back out. No extra code is required for this; it comes built into both `px.treemap()` and `px.sunburst()`.

In [ ]:
fig = px.treemap(
    full_hierarchy,
    path=["region", "category", "sub_category"],
    values="total_sales",
    color="category",
    title="Click a region or category to drill in; click the breadcrumb to zoom back out"
)
fig.show()

Try clicking into a single region, then a single category within it. Notice the rectangles resize to fill the available space at each zoomed-in level, since the chart is now only showing that branch of the hierarchy. This kind of drill-down is often the fastest way to explore a hierarchy you don't know well yet: start at the top level, then follow whatever looks interesting deeper in, without writing a new filter for every step.



## On your own

Using the same `sales` dataset, build a treemap or sunburst chart (your choice) with the hierarchy `region -> segment`, valued by any other metric instead of `sales`. In a markdown cell below, explain which chart type you chose and why it was the better fit for exploring *profit* by region and segment specifically, not just any hierarchy.

In [1]:
#TODO: Complete challenge here

**Type your Explanation of your design choices here**


## Submission

Export this notebook to HTML (`Jupyter: Export to HTML` from the Command Palette, or the "..." menu in the notebook toolbar) and submit the HTML file on Canvas.


## Where to go next: network visualization

Hierarchies are one kind of relational structure, where every item has exactly one parent. Many real datasets have relationships that don't fit that constraint: a social network where anyone can connect to anyone else, a citation graph, a set of flights connecting airports in every direction. Structures like that call for **network visualization** instead, typically built in Python with the `networkx` library for representing the graph and Plotly or a dedicated tool like `pyvis` for rendering it.

Network visualization is genuinely its own topic, with its own layout algorithms and its own set of design decisions, more than we can responsibly add to this course. If it interests you, two starting points:

- `networkx` documentation: <https://networkx.org/documentation/stable/>
- Plotly's network graph tutorial: <https://plotly.com/python/network-graphs/>
